In [6]:
import sys
from pyprojroot import here
sys.path.insert(0, str(here()))

In [7]:
import copy
import optuna
import numpy as np
from transformers import AutoTokenizer
import time

from src.train import train_single_fold, tokenize_df
from src.config import load_config
from src.data_holdout import get_pooled_df, get_fold_from_disk

import yaml
import torch
from datasets import Dataset
from pyprojroot import here
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)

from sklearn.metrics import f1_score
from peft import LoraConfig, get_peft_model, TaskType



In [8]:
MODEL_NAME = "microsoft/deberta-v3-base"

In [9]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)

linear_names = []
for name, module in model.named_modules():
    if module.__class__.__name__ == "Linear":
        linear_names.append(name)

for n in linear_names:
    print(n)

Loading weights: 100%|██████████| 198/198 [00:00<00:00, 3594.65it/s, Materializing param=deberta.encoder.rel_embeddings.weight]                     
DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
pooler.dense.weight                     | MI

deberta.encoder.layer.0.attention.self.query_proj
deberta.encoder.layer.0.attention.self.key_proj
deberta.encoder.layer.0.attention.self.value_proj
deberta.encoder.layer.0.attention.output.dense
deberta.encoder.layer.0.intermediate.dense
deberta.encoder.layer.0.output.dense
deberta.encoder.layer.1.attention.self.query_proj
deberta.encoder.layer.1.attention.self.key_proj
deberta.encoder.layer.1.attention.self.value_proj
deberta.encoder.layer.1.attention.output.dense
deberta.encoder.layer.1.intermediate.dense
deberta.encoder.layer.1.output.dense
deberta.encoder.layer.2.attention.self.query_proj
deberta.encoder.layer.2.attention.self.key_proj
deberta.encoder.layer.2.attention.self.value_proj
deberta.encoder.layer.2.attention.output.dense
deberta.encoder.layer.2.intermediate.dense
deberta.encoder.layer.2.output.dense
deberta.encoder.layer.3.attention.self.query_proj
deberta.encoder.layer.3.attention.self.key_proj
deberta.encoder.layer.3.attention.self.value_proj
deberta.encoder.layer.3.att

### Test query, value vector only vs all linear layers

In [5]:
# open shared settings
with open(here("configs/base.yaml"), "r") as f:
    base_data = yaml.full_load(f)

# get df
full_df = get_pooled_df()

# same pattern as collaborator: all folds
train_folds, val_folds = get_fold_from_disk(full_df)

# load tokenizer and write tokenize function
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fold(df):
    ds = Dataset.from_pandas(df, preserve_index=False)

    def _tokenize(examples):
        return tokenizer(
            examples["abstract_conclusion"],
            examples["press_release_conclusion"],
            truncation=True,
            padding="max_length",
            max_length=base_data["max_length"],
        )

    cols_to_remove = [c for c in ds.column_names if c != "exaggeration_label"]
    ds = ds.map(_tokenize, batched=True, remove_columns=cols_to_remove)
    ds = ds.rename_column("exaggeration_label", "labels")
    ds.set_format("torch")
    return ds

# tokenize all folds
train_ds = [tokenize_fold(df) for df in train_folds]
val_ds = [tokenize_fold(df) for df in val_folds]

for i in range(len(train_ds)):
    print(f"Fold {i}: train={len(train_ds[i])} val={len(val_ds[i])}")

# define metrics function
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"macro_f1": f1_score(labels, preds, average="macro")}


/workspace/exaggeration-detection/src/data_holdout.py:65: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .replace(ENCODED_LABELS)
Map: 100%|██████████| 106/106 [00:00<00:00, 8747.25 examples/s]

Fold 0: train=424 val=106
Fold 1: train=424 val=106
Fold 2: train=424 val=106
Fold 3: train=424 val=106
Fold 4: train=424 val=106


In [ ]:
def run_quick_lora_test(target_modules, fold=0):
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=3
    )

    peft_config = LoraConfig(
        task_type=TaskType.SEQ_CLS,
        target_modules=target_modules,
        r=8,
        lora_alpha=16,
        lora_dropout=0.05,
        bias="none",
    )

    model = get_peft_model(model, peft_config)
    model.print_trainable_parameters()

    training_args = TrainingArguments(
        output_dir=str(here("results/tmp_lora_test_deberta")),
        eval_strategy="epoch",
        save_strategy="no",
        learning_rate=2e-4,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        weight_decay=0.01,
        warmup_ratio=0.1,
        num_train_epochs=5,
        metric_for_best_model="macro_f1",
        fp16=True,
        seed=42,
        logging_steps=10,
        report_to="none",
        skip_memory_metrics=True,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_ds[fold],
        eval_dataset=val_ds[fold],
        compute_metrics=compute_metrics,
    )

    start = time.time()
    trainer.train()
    metrics = trainer.evaluate()
    elapsed = time.time() - start

    del trainer, model
    torch.cuda.empty_cache()

    return {
        "target_modules": target_modules,
        "runtime_min": elapsed / 60,
        "macro_f1": metrics["eval_macro_f1"],
    }

results_qv = run_quick_lora_test(["query","value"])
results_all = run_quick_lora_test("all-linear")

print(results_qv)
print(results_all)

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 1428.80it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]              
RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consid

trainable params: 887,811 || all params: 125,535,750 || trainable%: 0.7072


Epoch,Training Loss,Validation Loss,Macro F1
1,0.983513,0.931947,0.253411
2,0.850388,0.937608,0.253411
3,0.913065,0.920926,0.253411
4,0.865106,0.921930,0.253411
5,1.027179,0.924406,0.253411


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 1574.35it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]              
RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consid

trainable params: 1,920,003 || all params: 126,567,942 || trainable%: 1.5170


Epoch,Training Loss,Validation Loss,Macro F1
1,0.922802,0.940363,0.253411
2,0.876832,0.944502,0.253411
3,0.981848,0.930095,0.253411
4,0.838548,0.947200,0.253411
5,0.987828,0.946294,0.328457


{'target_modules': ['query', 'value'], 'runtime_min': 0.6380618373552959, 'macro_f1': 0.253411306042885}
{'target_modules': 'all-linear', 'runtime_min': 1.073991020520528, 'macro_f1': 0.3284566838783706}


## Optuna Experiment

In [11]:

# open shared settings
with open(here("configs/base.yaml"), "r") as f:
    base_data = yaml.full_load(f)

# get df
full_df = get_pooled_df()

# same pattern as collaborator: all folds
train_folds, val_folds = get_fold_from_disk(full_df)

# load tokenizer and write tokenize function
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fold(df):
    ds = Dataset.from_pandas(df, preserve_index=False)

    def _tokenize(examples):
        return tokenizer(
            examples["abstract_conclusion"],
            examples["press_release_conclusion"],
            truncation=True,
            padding="max_length",
            max_length=base_data["max_length"],
        )

    cols_to_remove = [c for c in ds.column_names if c != "exaggeration_label"]
    ds = ds.map(_tokenize, batched=True, remove_columns=cols_to_remove)
    ds = ds.rename_column("exaggeration_label", "labels")
    ds.set_format("torch")
    return ds

# tokenize all folds
train_ds = [tokenize_fold(df) for df in train_folds]
val_ds = [tokenize_fold(df) for df in val_folds]

for i in range(len(train_ds)):
    print(f"Fold {i}: train={len(train_ds[i])} val={len(val_ds[i])}")

# define metrics function
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"macro_f1": f1_score(labels, preds, average="macro")}


# set up Optuna objective
def objective(trial):
    folds_to_run = [0, 1, 3] # folds chosen based on previous experiments

    # LoRA-specific search space
    lr = trial.suggest_float("learning_rate", 1e-5, 5e-4, log=True)
    batch_size = trial.suggest_categorical("per_device_train_batch_size", [8])
    weight_decay = trial.suggest_float("weight_decay", 0.0, 0.1)
    warmup_ratio = trial.suggest_float("warmup_ratio", 0.05, 0.2)
    num_epochs = 8

    lora_r = trial.suggest_categorical("lora_r", [2, 4, 8, 16])
    lora_alpha = trial.suggest_categorical("lora_alpha", [4, 8, 16, 32])
    lora_dropout = trial.suggest_categorical("lora_dropout", [0.0, 0.05, 0.1])

    f1_scores = []
    fold_times_min = []

    for fold in folds_to_run:
        fold_start = time.time()
        model = AutoModelForSequenceClassification.from_pretrained(
            MODEL_NAME,
            num_labels=3
        )

        peft_config = LoraConfig(
            task_type=TaskType.SEQ_CLS,
            target_modules=["query_proj", "key_proj", "value_proj", "dense"], #approximation of all linear, get same results  in testing
            r=lora_r,
            lora_alpha=lora_alpha,
            lora_dropout=lora_dropout,
            bias="none",
        )

        model = get_peft_model(model, peft_config)

        # confirm correct number of trainable parameters
        if fold == 0:
            model.print_trainable_parameters()

        training_args = TrainingArguments(
            output_dir=str(here(f"results/optuna/deberta_lora2/trial-{trial.number}/fold-{fold}")),
            eval_strategy=base_data["eval_strategy"],
            save_strategy="no",              # no checkpoints this time
            learning_rate=lr,
            per_device_train_batch_size=batch_size,
            per_device_eval_batch_size=batch_size,
            weight_decay=weight_decay,
            warmup_ratio=warmup_ratio,
            num_train_epochs=num_epochs,
            metric_for_best_model="macro_f1",
            fp16=False,
            seed=base_data["seed"],
            logging_steps=10,
            report_to=base_data["report_to"],
            skip_memory_metrics=True,
        )

        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=train_ds[fold],
            eval_dataset=val_ds[fold],
            compute_metrics=compute_metrics,
            callbacks=[EarlyStoppingCallback(early_stopping_patience=10)],
        )

        trainer.train()
        eval_metrics = trainer.evaluate()
        f1_scores.append(eval_metrics["eval_macro_f1"])


        fold_time_min = (time.time() - fold_start) / 60
        fold_times_min.append(fold_time_min)
        print(
            f"Trial {trial.number} | Fold {fold} | "
            f"F1={eval_metrics['eval_macro_f1']:.4f} | "
            f"Time={fold_time_min:.2f} min"
        )

        del trainer
        del model
        torch.cuda.empty_cache()

    return float(np.mean(f1_scores))

# study definition
db_path = here("results/optuna/optuna_study_deberta_lora2.db")
storage_url = f"sqlite:///{db_path}"
n_startup_trials = 5

print(storage_url)

study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(
        n_startup_trials=n_startup_trials,
        seed=42,
    ),
    study_name="deberta-lora-hyperparameter-search2",
    storage=storage_url,
    load_if_exists=True
)

study.optimize(objective, n_trials=50)

# best trials
print(f"\n{'='*60}")
print(f"  Best macro_f1: {study.best_trial.value:.4f}")
print(f"  Best params:   {study.best_trial.params}")
print(f"{'='*60}")

/workspace/exaggeration-detection/src/data_holdout.py:65: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .replace(ENCODED_LABELS)
Map: 100%|██████████| 106/106 [00:00<00:00, 10995.06 examples/s]


Fold 0: train=424 val=106
Fold 1: train=424 val=106
Fold 2: train=424 val=106
Fold 3: train=424 val=106
Fold 4: train=424 val=106
sqlite:////workspace/exaggeration-detection/results/optuna/optuna_study_deberta_lora2.db


[I 2026-03-23 22:32:59,530] A new study created in RDB with name: deberta-lora-hyperparameter-search2
Loading weights: 100%|██████████| 198/198 [00:00<00:00, 2977.79it/s, Materializing param=deberta.encoder.rel_embeddings.weight]                     
DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED |

trainable params: 337,155 || all params: 184,761,606 || trainable%: 0.1825


Using EarlyStoppingCallback without load_best_model_at_end=True. Once training is finished, the best model will not be loaded automatically.


...

Trial 49 | Fold 1 | F1=0.2534 | Time=1.14 min


Loading weights: 100%|██████████| 198/198 [00:00<00:00, 4177.95it/s, Materializing param=deberta.encoder.rel_embeddings.weight]                     
DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
pooler.dense.weight                     | MI

Epoch,Training Loss,Validation Loss,Macro F1
1,0.962424,0.945146,0.253411
2,1.042560,0.935830,0.253411
3,0.957304,0.959111,0.253411
4,0.835918,0.946629,0.253411
5,1.043024,0.937579,0.253411
6,0.925288,0.945679,0.253411
7,0.818270,0.940882,0.253411
8,0.992709,0.941507,0.253411


[I 2026-03-24 01:23:03,371] Trial 49 finished with value: 0.253411306042885 and parameters: {'learning_rate': 0.00011841298581918639, 'per_device_train_batch_size': 8, 'weight_decay': 0.09433207124610177, 'warmup_ratio': 0.1970333647733592, 'lora_r': 2, 'lora_alpha': 8, 'lora_dropout': 0.05}. Best is trial 6 with value: 0.28204680015814715.


Trial 49 | Fold 3 | F1=0.2534 | Time=1.14 min

  Best macro_f1: 0.2820
  Best params:   {'learning_rate': 1.0609624330107086e-05, 'per_device_train_batch_size': 8, 'weight_decay': 0.06135622195040731, 'warmup_ratio': 0.19204428145852243, 'lora_r': 2, 'lora_alpha': 8, 'lora_dropout': 0.1}


In [12]:

print(study.best_trial)
print(study.best_value)
print(study.best_params)

FrozenTrial(number=6, state=<TrialState.COMPLETE: 1>, values=[0.28204680015814715], datetime_start=datetime.datetime(2026, 3, 23, 22, 53, 7, 319803), datetime_complete=datetime.datetime(2026, 3, 23, 22, 56, 32, 865622), params={'learning_rate': 1.0609624330107086e-05, 'per_device_train_batch_size': 8, 'weight_decay': 0.06135622195040731, 'warmup_ratio': 0.19204428145852243, 'lora_r': 2, 'lora_alpha': 8, 'lora_dropout': 0.1}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'learning_rate': FloatDistribution(high=0.0005, log=True, low=1e-05, step=None), 'per_device_train_batch_size': CategoricalDistribution(choices=(8,)), 'weight_decay': FloatDistribution(high=0.1, log=False, low=0.0, step=None), 'warmup_ratio': FloatDistribution(high=0.2, log=False, low=0.05, step=None), 'lora_r': CategoricalDistribution(choices=(2, 4, 8, 16)), 'lora_alpha': CategoricalDistribution(choices=(4, 8, 16, 32)), 'lora_dropout': CategoricalDistribution(choices=(0.0, 0.05, 0.1))}, trial_i